# Header

**NB:** You need 120g to run this entire notebook!

In [ ]:
import os
from scipy import signal, stats
import numpy as np

# os.environ["MODIN_ENGINE"] = "ray"  # Modin will use Ray
# import modin.pandas as pd
import pandas as pd

import bisect
import importlib
import warnings
from pathlib import Path
import gc

# from tqdm.notebook import trange, tqdm
# # Create new `pandas` methods which use `tqdm` progress
# tqdm.pandas()

import ete3
np.random.seed(7)

# import psutil, os
# def mem():
#     return psutil.Process(os.getpid()).memory_info().rss / 1024**3

Plotting setup:

In [ ]:
%matplotlib inline

# Make inline plots vector graphics instead of raster graphics
from matplotlib_inline.backend_inline import set_matplotlib_formats
#set_matplotlib_formats('pdf', 'svg')
set_matplotlib_formats('retina', 'png')

import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D

#matplotlib.rcParams['figure.figsize'] = (20.0, 10.0)

# import mpld3

import seaborn as sns
sns.set() # sets seaborn default "prettyness:
sns.set_style("white")
sns.set_context("notebook")

# lowess for plotting
from statsmodels.nonparametric.smoothers_lowess import lowess

# My own paired palette replacing the last brown pair with violets
sns.color_palette('Paired').as_hex()
Paired = sns.color_palette(['#a6cee3', '#1f78b4', '#b2df8a', '#33a02c', '#fb9a99', '#e31a1c',
                            '#fdbf6f', '#ff7f00', '#cab2d6','#6a3d9a', '#e585cf', '#ad009d'])
#sns.palplot(Paired)

from shared_vars_and_func import *

Monospace font for numbers in tables:

In [ ]:
%%html
<style> table { font-variant-numeric: tabular-nums; } </style>

In [ ]:
remapped_data_path = Path('../data/composition_cpg_remapped')

## Load species details

In [ ]:
species_details = pd.read_csv('../data/species_details.txt', sep='\t')
species_details.head()

# CGI data

In [ ]:
cgi = pd.read_csv('../data/bed/CGI-taeGut1.txt', sep='\t')
cgi['pos'] = (cgi.start + (cgi.end - cgi.start) / 2).round().astype(int)
cgi['chrom'] = cgi['chr'].str.replace('chr', '')
cgi.head()

# Find size order of TAEGU chromosomes

In [ ]:
from Bio import SeqIO
import gzip

records = []
with gzip.open('../../../../data/assembly/Taeniopygia_guttata.fa.gz', 'rt') as f:
    for record in SeqIO.parse(f, "fasta"):
        if record.id in chromosomes:
            records.append([record.id, len(record)])
taegu_chromosome_sizes = (pd.DataFrame()
                          .from_records(records, columns=['chrom', 'chrom_size'])
                          .sort_values(by='chrom_size', ascending=False)
                         )
taegu_chromosome_sizes

In [ ]:
taegu_chromosome_sizes.to_hdf('../results/taegu_chromosome_sizes.h5', key='df', format='table')

# Hotspot data

In [ ]:
hotspots = pd.read_csv('../data/bed/hotspots.bed', names=['chrom', 'start', 'end'], sep='\t')
hotspots['pos'] = (hotspots.start + (hotspots.end - hotspots.start) / 2).round().astype(int)
hotspots['chrom'] = hotspots.chrom.str.replace('chr', '')
hotspots.rename(columns={'stop': 'end'}, inplace=True)
hotspots.head()

In [ ]:
hotspots.pos.unique().size

Function for reducing memory of data frames:

# Data mapped relative to hotspots (hotspot_data)

In [ ]:
branch_lengths = pd.read_hdf('../results/branch_lengths.h5')

All 1kb windows overlapping hotspot center are set to zero distance. For all other windows, the distance is the distane to the closest hotspot center.

In [ ]:
df_list = list()
for path in remapped_data_path.iterdir():
    if path.name.endswith('.hotspot_relative.txt'):
        df = pd.read_csv(str(path), sep='\t', low_memory=False)
        df_list.append(df)
hotspot_table = pd.concat(df_list, sort=True).merge(branch_lengths, on=['species', 'ancestor'], how='left')

In [ ]:
hotspot_table.to_hdf('../results/hotspot_table.h5', key='df', format='table')

In [ ]:
del df_list
gc.collect()

Optimize data frame. Exclude one of the eagles. We also exclude windows smaller than 1000. These are split windows overlapping midpoint between two hotspots. 

In [ ]:
def data_filter(df):
    include_species = ~df.species.isin(['HALAL'])
    win_size = (df.end - df.start).abs()
    include_window = (win_size == 1000) | (win_size == 0)
    return include_species & include_window

hotspot_data = optimize_data_frame(hotspot_table).loc[data_filter]
hotspot_data['species'] = hotspot_data['species'].cat.remove_unused_categories()

In [ ]:
len(hotspot_table.start_prox.unique())

In [ ]:
del hotspot_table
gc.collect()

In [ ]:
hotspot_data['hotspot_center'] = (hotspot_data.start_prox + (hotspot_data.end_prox - hotspot_data.start_prox) / 2).astype('int32')

Add species names:

In [ ]:
hotspot_data['species_code'] = hotspot_data.species
name_mapping = dict(zip(species_details.species2014, species_details.english))
hotspot_data['species'] = [name_mapping[x] for x in hotspot_data.species_code]

hotspot_data['species'] = hotspot_data['species'].astype('category')

Compute the number of aligned bases in each window:

In [ ]:
hotspot_data['nr_aligned'] = hotspot_data[base_counts].sum(axis=1)

Compute substitution rates across each external branch by dividing substitution counts by the ancestral count of the base substituted from:

In [ ]:
for subst in substitution_counts:
    from_base, to_base = subst[1], subst[3]
    
    adding_subst = [x for x in substitution_counts if x[3] == from_base]
    removing_subst = [x for x in substitution_counts if x[1] == from_base]
    
    hotspot_data[subst.replace('n', 'r')] = hotspot_data[subst] / \
        (hotspot_data['n' + from_base] - hotspot_data[adding_subst].sum(axis=1) + hotspot_data[removing_subst].sum(axis=1))

Normalize so rates are relative to so they are relative to the 30kb - 40kb flanks:

In [ ]:
# flank_rates = (hotspot_data.loc[(hotspot_data.bin.abs() > flank_start) & (hotspot_data.bin.abs() <= 40000)]
#                .groupby(['species_code'], observed=False)
#                [substitutions]
#                .mean()
#               )
# df = hotspot_data.merge(flank_rates, on='species_code', how='left', suffixes=('_raw', '_flank_mean'))
# for pattern in substitutions:
#     hotspot_data[pattern] = df[pattern + '_raw'] /  df[pattern + '_flank_mean']

In [ ]:
len(hotspot_data.species.unique())

In [ ]:
hotspot_data.head()

In [ ]:
hotspot_data.to_hdf('../results/hotspot_data.h5', key='df', format='table')

Make a copy of the entire data frame and do the same but including CpG sites:

In [ ]:
hotspot_data_incl_cpg = hotspot_data.copy(deep=True)

for subst in substitution_counts:
    from_base, to_base = subst[1], subst[3]
    
    adding_subst = [x for x in substitution_counts+cpg_substitution_counts if x[3].upper() == from_base]
    removing_subst = [x for x in substitution_counts+cpg_substitution_counts if x[1].upper() == from_base]
    
    same_subst = [x for x in substitution_counts+cpg_substitution_counts if x[1:].upper() == subst[1:]]
   
    from_base_count = hotspot_data['n' + from_base]
    if from_base in 'GC':
        from_base_count += hotspot_data['n' + from_base.lower()]

    hotspot_data_incl_cpg[subst.replace('n', 'r')] = hotspot_data[same_subst].sum(axis=1) / \
            (from_base_count - hotspot_data[adding_subst].sum(axis=1) + hotspot_data[removing_subst].sum(axis=1))    

Normalize so rates are relative to so they are relative to the 30kb - 40kb flanks:

In [ ]:
# flank_rates_incl_cpg = (hotspot_data_incl_cpg.loc[(hotspot_data_incl_cpg.bin.abs() > flank_start) & (hotspot_data_incl_cpg.bin.abs() <= 40000)]
#                .groupby(['species_code'], observed=False)
#                [substitutions]
#                .mean()
#               )
# df = hotspot_data_incl_cpg.merge(flank_rates_incl_cpg, on='species_code', how='left', suffixes=('_raw', '_flank_mean'))
# for pattern in substitutions:
#     hotspot_data_incl_cpg[pattern] = df[pattern + '_raw'] /  df[pattern + '_flank_mean']

In [ ]:
hotspot_data_incl_cpg.head()

In [ ]:
hotspot_data_incl_cpg.to_hdf('../results/hotspot_data_incl_cpg.h5', key='df', format='table')

In [ ]:
del hotspot_data
gc.collect()

In [ ]:
del hotspot_data_incl_cpg
gc.collect()

# Long format hotspot data (subst_data)

Make a long format data frames with substitution rate data within 40kb from each hotspot:

In [ ]:
hotspot_data = pd.read_hdf('../results/hotspot_data.h5')

In [ ]:
id_cols = ['species', 'species_code', 'ancestor', 'hotspot_center', 'bin', 'branch_length', 'ultra_branch_length', 'nr_aligned']
subst_data = (hotspot_data
              .loc[lambda df: df.bin.abs() <= 40000, id_cols + substitutions + base_counts]
              .melt(id_vars=id_cols,
                    value_vars=substitutions,
                    var_name='pattern', value_name='rate')
             )
subst_data['pattern'] = pd.Categorical(subst_data.pattern.values, 
                                       categories=substitutions, ordered=True)
subst_data.loc[subst_data.pattern.isin(transitions), 'type'] = 'Transition'
subst_data.loc[~subst_data.pattern.isin(transitions), 'type'] = 'Transversion'
subst_data['type'] = pd.Categorical(subst_data.type.values, categories=['Transition', 'Transversion'], ordered=True)

# scale with flank means
subst_data = (subst_data
 .loc[subst_data.bin.abs() > flank_start]
 .groupby(['species_code', 'pattern'], observed=False)
 .rate
 .mean()
 .reset_index()
 .rename(columns={'rate':'flank_rate'}) 
).merge(subst_data, on=['species_code', 'pattern'], how='right')
subst_data['rate'] /= subst_data['flank_rate']
subst_data.head()

In [ ]:
# (subst_data
#  .loc[subst_data.bin.abs() > flank_start]
#  .groupby(['species_code', 'pattern'], observed=False)
#  .rate
#  .mean()
#  .reset_index()
#  .rename(columns={'rate':'flank_rate'}) 
# ).merge(subst_data, on=['species_code', 'pattern'], how='right')

# (subst_data
#  .rename(columns={'rate':'raw_rate'})
#  .merge(mean_flank_rates, on=['species_code', 'pattern'], how='left')
# )                                                            

In [ ]:

# (subst_data
#  .head()
#  .groupby(['species_code', 'pattern'], observed=False)
#  .apply(lambda df: df['rate']/df.loc[subst_data.bin.abs() > flank_start, 'rate'])
#  .reset_index()
# )

-----

## Number of aligned bases and filtering

Distribution of number of aligned bases in windows for TAEGU:

In [ ]:
plt.hist(subst_data.loc[subst_data.species_code == 'TAEGU'].nr_aligned, bins=30)
sns.despine()
plt.savefig(figures_path / 'distr_nr_aligned_bases.pdf')

Distribution of number of aligned bases in windows for all others:

In [ ]:
with sns.axes_style('whitegrid'):
    g = sns.FacetGrid(subst_data.loc[subst_data.species_code != 'TAEGU'], col='species', col_wrap=10, sharey=True)
    g.map(plt.hist, 'nr_aligned', bins=30).add_legend()
    g.set_titles(row_template = '{row_name}', col_template = '{col_name}') ;
    plt.savefig(figures_path / 'distr_nr_aligned_bases_indiv_species.pdf')    

In [ ]:
with sns.axes_style('whitegrid'):
    g = sns.FacetGrid(subst_data,
                      col='species', col_wrap=10, sharey=True)
    g.map(sns.lineplot, 'bin', 'nr_aligned', errorbar=None).add_legend()
    g.set_titles(row_template = '{row_name}', col_template = '{col_name}') ;
    [ax.set_ylim(bottom=0) for ax in g.axes.flat]
    plt.savefig(figures_path / 'nr_aligned_base_across_hotspots_indiv_species.pdf')        

In [ ]:
with sns.axes_style('whitegrid'):
    g = sns.FacetGrid(subst_data.loc[subst_data.nr_aligned >= 900],
                      col='species', col_wrap=10, sharey=True)
    g.map(sns.lineplot, 'bin', 'nr_aligned', errorbar=None).add_legend()
    g.set_titles(row_template = '{row_name}', col_template = '{col_name}') ;
    [ax.set_ylim(bottom=0) for ax in g.axes.flat]
    plt.savefig(figures_path / 'nr_aligned_base_across_hotspots_indiv_species_only_min900bases.pdf')            

> It seems there are important differences in the alignemnt quality in the different species. Especially with some of the long branches. 

> We will be asked how much of the differences we find are due to misalignemnts. To that we can say that any misalignment will inflate all substitution rates at hotspots not just a few. As we show further down, the A2T and T2A substitution rates are constant across the hotspots.

We use only windows where with at least 300 aligned bases. (below we also test if this cutoff matters to the patterns observed)

In [ ]:
removed_rows = subst_data[subst_data.nr_aligned < 300].index
subst_data.drop(removed_rows, inplace=True)

Normalize each substitution rate with a global mutation rate divided by the external branch length. 

By dividing by branch length we take variation in overall subsititution rate into account. This should make the summed substitituion rates outside hotspots should be the same across species.

In [ ]:
#subst_data['rate'] /= subst_data.branch_length

In [ ]:
optimize_data_frame(subst_data, inplace=True)
subst_data.head()

It seems one hotspot appears twice. Remove that one so positions remain unqiue across chromosomesÆ

In [ ]:
subst_data = subst_data.loc[subst_data.hotspot_center != 11341000]

In [ ]:
sorted(subst_data.species.unique().to_list())

In [ ]:
subst_data.to_hdf('../results/subst_data.h5', key='df', format='table')

And the same including CpG sites:

In [ ]:
hotspot_data_incl_cpg = pd.read_hdf('../results/hotspot_data_incl_cpg.h5')

In [ ]:
subst_data_incl_cpg = (hotspot_data_incl_cpg
              .loc[lambda df: df.bin.abs() <= 40000, id_cols + substitutions + base_counts]
              .melt(id_vars=id_cols,
                    value_vars=substitutions,
                    var_name='pattern', value_name='rate')
             )
subst_data_incl_cpg['pattern'] = pd.Categorical(subst_data_incl_cpg.pattern.values, 
                                       categories=substitutions, ordered=True)
subst_data_incl_cpg.loc[subst_data_incl_cpg.pattern.isin(transitions), 'type'] = 'Transition'
subst_data_incl_cpg.loc[~subst_data_incl_cpg.pattern.isin(transitions), 'type'] = 'Transversion'
subst_data_incl_cpg['type'] = pd.Categorical(subst_data_incl_cpg.type.values, categories=['Transition', 'Transversion'], ordered=True)

# scale with flank means
subst_data_incl_cpg = (subst_data_incl_cpg
 .loc[subst_data_incl_cpg.bin.abs() > flank_start]
 .groupby(['species_code', 'pattern'], observed=False)
 .rate
 .mean()
 .reset_index()
 .rename(columns={'rate':'flank_rate'}) 
).merge(subst_data_incl_cpg, on=['species_code', 'pattern'], how='right')
subst_data_incl_cpg['rate'] /= subst_data_incl_cpg['flank_rate']
subst_data_incl_cpg.head()

We use only windows where with at least 300 aligned bases. (below we also test if this cutoff matters to the patterns observed)

In [ ]:
removed_rows = subst_data_incl_cpg[subst_data_incl_cpg.nr_aligned < 300].index
subst_data_incl_cpg.drop(removed_rows, inplace=True)

Normalize each substitution rate with a global mutation rate divided by the external branch length. 

By dividing by branch length we take variation in overall subsititution rate into account. This should make the summed substitituion rates outside hotspots should be the same across species.

In [ ]:
#subst_data_incl_cpg['rate'] /= subst_data_incl_cpg.branch_length

In [ ]:
optimize_data_frame(subst_data_incl_cpg, inplace=True)
subst_data_incl_cpg.head()

It seems one hotspot appears twice...

In [ ]:
subst_data_incl_cpg = subst_data_incl_cpg.loc[subst_data_incl_cpg.hotspot_center != 11341000]

In [ ]:
subst_data_incl_cpg.to_hdf('../results/subst_data_incl_cpg.h5', key='df', format='table')

In [ ]:
del hotspot_data
del hotspot_data_incl_cpg
gc.collect()

See if the summed substitituion rates outside hotspots are the same across species:

In [ ]:
mean_flank_rates = (subst_data
                    .loc[subst_data.bin.abs() > flank_start]
                    .groupby('species', observed=False)
                    .rate
                    .mean()
                    .reset_index()
                    .rename(columns={'rate': 'mean_flank_rate'})
                    )
with sns.plotting_context('notebook', font_scale=0.8):
    fig, ax = plt.subplots(figsize=(8, 8))
    ax = sns.barplot(y="species", x="mean_flank_rate", data=mean_flank_rates, color='lightgrey')
    plt.savefig(figures_path / 'mean_norm_flank_subst_rates.pdf')                

# Pivot table with column for each pattern (rate_table)

Rates are means across windows in the same position across hotspots.

In [ ]:
subst_data.species.cat.categories

In [ ]:
rate_table = pd.pivot_table(subst_data, observed=False,
                            index=['species', 'species_code', 'bin'], columns='pattern', values='rate')

rate_table.columns = rate_table.columns.astype('str')
for a, b in paired_patterns:
    rate_table[f'{a}-{b}'] = rate_table[a] - rate_table[b]
rate_table.head()

In [ ]:
rate_table.reset_index().species.cat.categories

In [ ]:
rate_table.to_hdf('../results/rate_table.h5', key='df', format='table')

In [ ]:
rate_table_incl_cpg = pd.pivot_table(subst_data_incl_cpg, observed=False,
                            index=['species', 'species_code', 'bin'], columns='pattern', values='rate')
rate_table_incl_cpg.columns = rate_table_incl_cpg.columns.astype('str')
for a, b in paired_patterns:
    rate_table_incl_cpg[f'{a}-{b}'] = rate_table_incl_cpg[a] - rate_table_incl_cpg[b]
rate_table_incl_cpg.head()

In [ ]:
rate_table_incl_cpg.to_hdf('../results/rate_table_incl_cpg.h5', key='df', format='table')

In [ ]:
del rate_table
del rate_table_incl_cpg
gc.collect()

# Mean GC* for each chromosome of each species

In [ ]:
records = []
for species in species_details.species:
    if species in ['HALAL']:
        continue
    for chrom in chromosomes:
        f_name = f'/home/kmt/Birds/faststorage/data/composition/{species}/chr{chrom}.ancestor.composition.1000.txt'
        try:
            df = pd.read_csv(f_name, sep='\t')
            nA, nT, nG, nC = df.nA_tot.sum(), df.nT_tot.sum(), df.nG_tot.sum(), df.nC_tot.sum()
            GCcontent = (nG + nC) / (nA + nT + nG + nC)
            GCflux = (df.nA2G.sum()/nA + df.nA2C.sum()/nA + df.nT2G.sum()/nT + df.nT2C.sum()/nT) / \
                     (df.nC2A.sum()/nC + df.nC2T.sum()/nC + df.nG2A.sum()/nG + df.nG2T.sum()/nG)
            GCstar = GCflux/(1 + GCflux)
            records.append([species, chrom, GCstar, GCflux, GCcontent])
        except:
            pass
            #print('missing:', f_name)

In [ ]:
chromosome_GCstar = pd.DataFrame.from_records(records, columns=['species_code', 'chrom', 'GCstar', 'GCflux', 'GCcontent'])

chromosome_GCstar = (chromosome_GCstar
                     .merge(species_details.rename(columns={'species': 'species_code'})[['species_code', 'english']], 
                            on='species_code', how='left')
                     .rename(columns={'english': 'species'})
                    )

chromosome_GCstar.head()

In [ ]:
chromosome_GCstar.to_hdf('../results/chromosome_GCstar.h5', key='df')

In [ ]:
del chromosome_GCstar
gc.collect()

# Read hotspot distances relative to CGI, TSS and TES

## CGI relative data

In [ ]:
df_list = list()
for path in remapped_data_path.iterdir():
    if path.name.endswith('.cgi_relative.txt'):
        df = pd.read_csv(str(path), sep='\t', low_memory=False)
        df_list.append(df)
cgi_table = pd.concat(df_list, sort=True).merge(branch_lengths, on=['species', 'ancestor'], how='left')

optimize_data_frame(cgi_table).to_hdf(results_path / 'cgi_data.h5', key='df', mode='w', format='table')

In [ ]:
del cgi_table
gc.collect()

## Promoter relative data

In [ ]:
df_list = list()
for path in remapped_data_path.iterdir():
    if path.name.endswith('.promoter_relative.txt'):
        df = pd.read_csv(str(path), sep='\t', low_memory=False)
        df_list.append(df)
promoter_table = pd.concat(df_list, sort=True).merge(branch_lengths, on=['species', 'ancestor'], how='left')

optimize_data_frame(promoter_table).to_hdf(results_path / 'promoter_data.h5', key='df', mode='w', format='table')

In [ ]:
del promoter_table
gc.collect()

## TSS relative data

In [ ]:
df_list = list()
for path in remapped_data_path.iterdir():
    if path.name.endswith('.tss_relative.txt'):
        df = pd.read_csv(str(path), sep='\t', low_memory=False)
        df_list.append(df)
tss_table = pd.concat(df_list, sort=True).merge(branch_lengths, on=['species', 'ancestor'], how='left')

optimize_data_frame(tss_table).to_hdf(results_path / 'tss_data.h5', key='df', mode='w', format='table')

In [ ]:
del tss_table
gc.collect()

## TES relative data

In [ ]:
df_list = list()
for path in remapped_data_path.iterdir():
    if path.name.endswith('.tes_relative.txt'):
        df = pd.read_csv(str(path), sep='\t', low_memory=False)
        df_list.append(df)
tes_table = pd.concat(df_list, sort=True).merge(branch_lengths, on=['species', 'ancestor'], how='left')

optimize_data_frame(tes_table).to_hdf(results_path / 'tes_data.h5', key='df', mode='w', format='table')

In [ ]:
del tes_table
gc.collect()

# Distance to CGI, TSS, TES, and promoter (hotspot_anno_dist)

Hotspot distance to center of closest CGI, TSS, TES, and promoter:

In [ ]:
def read_dist_data(file_name):
    del_cols = ['pos', 'start', 'end', 'start_prox', 'end_prox', 'start_orig', 'end_orig']
    return (pd.read_csv(file_name, sep='\t')
            .assign(
                    hotspot_center=lambda df: (df.start_orig + (df.end_orig - df.start_orig) / 2).astype('int32'),
                    dist=lambda df: (df.start.abs()+(df.end.abs()-df.start.abs())/2).astype('int32'),
                   )
            .drop(del_cols, axis=1)
           )

remapped_data_path = data_path / 'composition_cpg_remapped'

def remove_duplicates(df):
    return df.sample(frac=1).reset_index(drop=True).drop_duplicates(subset=['chrom', 'hotspot_center'])

hotspot_rel_cgi_data = read_dist_data(remapped_data_path / 'hotspot_rel_cgi.txt')
hotspot_rel_promoter_data = read_dist_data(remapped_data_path / 'hotspot_rel_promoter.txt')
hotspot_rel_tss_data = read_dist_data(remapped_data_path / 'hotspot_rel_tss.txt')
hotspot_rel_tes_data = read_dist_data(remapped_data_path / 'hotspot_rel_tes.txt')

# There are duplicate rows for two reasons: 
# 1. a lot of hotspots overlap more than one cgi. 
# 2. a few hotspots ~50 are split because they overlap the midpoint between
# So in case of duplicates we keep only one at random.

hotspot_rel_cgi_data = remove_duplicates(hotspot_rel_cgi_data)
hotspot_rel_promoter_data = remove_duplicates(hotspot_rel_promoter_data)
hotspot_rel_tss_data = remove_duplicates(hotspot_rel_tss_data)
hotspot_rel_tes_data = remove_duplicates(hotspot_rel_tes_data)

hotspot_rel_cgi_data.rename(columns={'bin': 'bin_cgi', 'dist': 'dist_cgi'}, inplace=True)
hotspot_rel_promoter_data.rename(columns={'bin': 'bin_promoter', 'dist': 'dist_promoter'}, inplace=True)
hotspot_rel_tss_data.rename(columns={'bin': 'bin_tss', 'dist': 'dist_tss'}, inplace=True)
hotspot_rel_tes_data.rename(columns={'bin': 'bin_tes', 'dist': 'dist_tes'}, inplace=True)

In [ ]:
merge_cols = ['chrom', 'hotspot_center']
hotspot_anno_dist = (hotspot_rel_cgi_data
                     .merge(hotspot_rel_tss_data)
                     .merge(hotspot_rel_tes_data)
                     .merge(hotspot_rel_promoter_data)
                     .sort_values(['chrom', 'hotspot_center'])
                    )
hotspot_anno_dist.head()

In [ ]:
hotspot_anno_dist.to_hdf('../results/hotspot_anno_dist.h5', key='df', format='table')

In [ ]:
del hotspot_rel_cgi_data
del hotspot_rel_promoter_data
del hotspot_rel_tss_data
del hotspot_rel_tes_data
del hotspot_anno_dist
gc.collect()

# GC*

## GC* of new mutations

Aggregate 1kb GC* in 200kb windows and record the 1 percentile GC* for each species:

In [ ]:
from chromwindow import window

@window(size=200000)
def window_gcstar(df):
    nA, nT, nG, nC = df.nA_tot.sum(), df.nT_tot.sum(), df.nG_tot.sum(), df.nC_tot.sum()
    if nA + nT + nG + nC < 100000:
        return np.nan
    toGC = df.nA2G.sum()/nA + df.nA2C.sum()/nA + df.nT2G.sum()/nT + df.nT2C.sum()/nT
    toAT = df.nC2A.sum()/nC + df.nC2T.sum()/nC + df.nG2A.sum()/nG + df.nG2T.sum()/nG
    if not toAT:
        return np.nan
    GCflux = toGC / toAT
    GCstar = GCflux/(1 + GCflux)
    return GCstar

records = []
big_df_list = []
for species, species_code in list(zip(species_details.english, species_details.species2014)):
    if species_code in ['HALAL']:
        continue
    df_list = []
    for chrom in chromosomes:
        f_name = f'/home/kmt/Birds/faststorage/data/composition/{species_code}/chr{chrom}.ancestor.composition.1000.txt'
        if not os.path.exists(f_name):
            print('Missing:', species_code, chrom)
            continue
        df = pd.read_csv(f_name, sep='\t')
        df['end'] = df.start + 1000
        df['chrom'] = chrom
        df['species'] = species
        df['species_code'] = species_code 
        df_list.append(df)
    if df_list:
        df = pd.concat(df_list).groupby(['chrom', 'species', 'species_code']).apply(window_gcstar).reset_index()
        records.append((species, species_code, df.window_gcstar.quantile(0.01)))
        big_df_list.append(df)
        
gcstar_200kb_windows = pd.concat(big_df_list)
gcstar_200kb_windows.head()

In [ ]:
gcstar_200kb_windows.to_hdf('../results/gcstar_200kb_windows.h5', 'df')

In [ ]:
one_percent_gcstar_quantiles = pd.DataFrame().from_records(records, columns=['species', 'species_code', 'GCstar_mut'])
one_percent_gcstar_quantiles.head()

In [ ]:
one_percent_gcstar_quantiles.to_hdf('../results/one_percent_gcstar_quantiles.h5', 'df')

In [ ]:
del gcstar_200kb_windows
del one_percent_gcstar_quantiles
gc.collect()

## GC* (gc_star_table)

GCflux, is calculated as the AT to GC over GC to AT substitution rate. The equilibrium GC content resulting from this bias, GC*, is calculated as GCflux/(1 + GCflux).


In [ ]:
def subst_gc_star(df):
    
    GCflux = (df.rA2G + df.rA2C + df.rT2G + df.rT2C) / \
             (df.rC2A + df.rC2T + df.rG2A + df.rG2T)
    GCstar = GCflux/(1 + GCflux)
    
    GCstar_TV = (df.rA2C + df.rT2G) / (df.rC2A + df.rG2T)
    GCstar_TV = GCstar_TV / (1 + GCstar_TV)

    GCstar_TS = (df.rA2G + df.rT2C) / \
                (df.rC2T + df.rG2A)
    GCstar_TS = GCstar_TS / (1 + GCstar_TS)
    
    return pd.DataFrame(dict(GCstar=GCstar, GCstar_TV=GCstar_TV, GCstar_TS=GCstar_TS))

In [ ]:
rate_table = pd.read_hdf('../results/rate_table.h5')

In [ ]:
gc_star_table = rate_table.groupby(['species', 'species_code', 'bin'], group_keys=False).apply(subst_gc_star).reset_index()
gc_star_table.head()

In [ ]:
gc_star_table.to_hdf('../results/gc_star_table.h5', key='df', format='table')

In [ ]:
del rate_table
gc.collect()

In [ ]:
rate_table_incl_cpg = pd.read_hdf('../results/rate_table_incl_cpg.h5')

In [ ]:
gc_star_table_incl_cpg = rate_table_incl_cpg.groupby(['species', 'species_code', 'bin'], group_keys=False).apply(subst_gc_star).reset_index()
gc_star_table_incl_cpg.head()

In [ ]:
gc_star_table_incl_cpg.to_hdf('../results/gc_star_table_incl_cpg.h5', key='df', format='table')

In [ ]:
del rate_table_incl_cpg
gc.collect()

## GC* long format (gc_star)

In [ ]:
gc_star = gc_star_table.melt(id_vars=['species', 'species_code', 'bin'], var_name='variable', value_name='frequency')
gc_star.to_hdf('../results/gc_star.h5', key='df', format='table')

In [ ]:
del gc_star
del gc_star_table
gc.collect()

In [ ]:
gc_star_incl_cpg = gc_star_table_incl_cpg.melt(id_vars=['species', 'species_code', 'bin'], var_name='variable', value_name='frequency')
gc_star_incl_cpg.to_hdf('../results/gc_star_incl_cpg.h5', key='df', format='table')

In [ ]:
del gc_star_incl_cpg
del gc_star_table_incl_cpg
gc.collect()

# Center vs flanks

## Mean species hotspot strength (hotspot_strengths)

In [ ]:
gc_star_table = pd.read_hdf('../results/gc_star_table.h5')
one_percent_gcstar_quantiles = pd.read_hdf('../results/one_percent_gcstar_quantiles.h5')

In [ ]:
hotspot_strengths = (pd.merge((gc_star_table.loc[gc_star_table.bin == 0]
                               .groupby(['species', 'species_code'], observed=True)
                               .GCstar.mean()
                               .to_frame('GCstar_hot')
                               .reset_index()
                              ),
                              (gc_star_table.loc[gc_star_table.bin.abs() >= flank_start]
                               .groupby(['species', 'species_code'], observed=True)
                               .GCstar.mean()
                               .to_frame('GCstar_flank')
                               .reset_index()
                              ),
                              on=['species', 'species_code']
                            )
                    .merge(one_percent_gcstar_quantiles, on=['species', 'species_code'], how='left')
                    )

from scipy.special import logit
hotspot_strengths['hotspot_strength'] = \
    (logit(hotspot_strengths.GCstar_hot) - logit(hotspot_strengths.GCstar_mut)) / \
    (logit(hotspot_strengths.GCstar_flank) - logit(hotspot_strengths.GCstar_mut))

hotspot_strengths['log_hotspot_strength'] = np.log2(hotspot_strengths['hotspot_strength'])

In [ ]:
hotspot_strengths.to_hdf('../results/hotspot_strengths.h5', key='df', format='table')

In [ ]:
del gc_star_table
del one_percent_gcstar_quantiles
del hotspot_strengths
gc.collect()

# THIS IS WHAT I WRONG I THINK: HOTSPOTS_RATE_TABLE IS MADE FROM SUBS_DATA, WHICH IS NOT SCALED WITH FLANK. IT SHOULD BE MADE FROM RATE_TABLE SOMEHOW or we could just normalize subst_data instead

## Strength of each hotspot (hotspots_rate_table)

In [ ]:
subst_data = pd.read_hdf('../results/subst_data.h5')

In [ ]:
df = (subst_data
 .loc[(subst_data.bin == 0) | (subst_data.bin.abs() >= flank_start)]
 .assign(is_center=lambda df: df.bin == 0)
 .groupby(['species', 'species_code', 'hotspot_center', 'pattern', 'is_center'])
 .rate.mean()
 .reset_index()
)

hotspots_rate_table = pd.pivot_table(df,
                            index=['species', 'species_code', 'hotspot_center', 'is_center'], columns='pattern', values='rate')
hotspots_rate_table.columns = hotspots_rate_table.columns.astype('str')
for a, b in paired_patterns:
    hotspots_rate_table[f'{a}-{b}'] = hotspots_rate_table[a] - hotspots_rate_table[b]
hotspots_rate_table.head()

In [ ]:
hotspots_rate_table.to_hdf('../results/hotspots_rate_table.h5', key='df', format='table')

In [ ]:
del subst_data
gc.collect()

<!-- ## ??? (hotspots_gc_star_table) -->

In [ ]:
# hotspots_gc_star_table = hotspots_rate_table.groupby(['species', 'species_code', 'is_center'], group_keys=False).apply(subst_gc_star).reset_index()
# hotspots_gc_star_table.head()

In [ ]:
# hotspots_gc_star_table.to_hdf('../results/hotspots_gc_star_table.h5', 'df', format='table')

In [ ]:
# del hotspots_rate_table
# del hotspots_gc_star_table
# gc.collect()

---

The stat function should be compute the hotspot_streangth

Pivot so that the GCstar_hot and GCstar_flank for each hostpos are in the same row. That the stat function can sample rows and compute hotspot_strength

In [ ]:
# hot_flank_gcstar = (pd.merge((hotspots_gc_star_table.loc[hotspots_gc_star_table.is_center]
#                                .groupby(['species', 'species_code', 'hotspot_center'], observed=True)
#                                .GCstar.mean()
#                                .to_frame('GCstar_hot')
#                                .reset_index()
#                               ),
#                               (hotspots_gc_star_table.loc[~hotspots_gc_star_table.is_center]
#                                .groupby(['species', 'species_code', 'hotspot_center'], observed=True)
#                                .GCstar.mean()
#                                .to_frame('GCstar_flank')
#                                .reset_index()
#                               ),
#                               on=['species', 'species_code', 'hotspot_center']
#                             )
#                     .merge(one_percent_gcstar_quantiles, on=['species', 'species_code'], how='left')
#                     )
# hot_flank_gcstar.head()

In [ ]:
# hot_flank_gcstar.to_hdf('../results/hot_flank_gcstar.h5', key='df', format='table')

---

## GC* at hotspot relative to flanks (gc_star_ratios, gc_star_ratio_table)

In [ ]:
gc_star = pd.read_hdf('../results/gc_star.h5')
gc_star.head()

In [ ]:
def get_gc_star_in_flanks(df):
    return df.loc[(df.bin.abs() < 40000) & (df.bin.abs() >= 10000)].groupby(['species', 'variable']).frequency.mean()

def get_gc_star_at_center(df):
    return df.loc[df.bin == 0].groupby(['species', 'variable']).frequency.mean()

def compute_gc_star_ratios(df):
    flank = get_gc_star_in_flanks(df)
    center = get_gc_star_at_center(df)
    ratios = (center / flank).reset_index().rename(columns={'frequency': 'ratio'})
    ratios_df = ratios.pivot(index='species', columns='variable', values='ratio').rename(columns=str).reset_index()
    ratios = ratios_df.melt(id_vars=['species'], var_name='variable', value_name='ratio')
    return ratios

gc_star_ratios = compute_gc_star_ratios(gc_star)
gc_star_ratios.head()

In [ ]:
gc_star_ratios.to_hdf('../results/gc_star_ratios.h5', key='df', format='table')

In [ ]:
gc_star_incl_cpg = pd.read_hdf('../results/gc_star_incl_cpg.h5')

In [ ]:
gc_star_ratios_incl_cpg = compute_gc_star_ratios(gc_star_incl_cpg)
gc_star_ratios_incl_cpg.head()

In [ ]:
gc_star_ratios_incl_cpg.to_hdf('../results/gc_star_ratios_incl_cpg.h5', key='df', format='table')

In [ ]:
gc_star_ratio_table = gc_star_ratios.pivot(index='species', columns='variable', values='ratio').reset_index()
gc_star_ratio_table.head()

In [ ]:
gc_star_ratio_table.to_hdf('../results/gc_star_ratio_table.h5', key='df', format='table')

In [ ]:
gc_star_ratio_table_incl_cpg = gc_star_ratios_incl_cpg.pivot(index='species', columns='variable', values='ratio').reset_index()
gc_star_ratio_table_incl_cpg.head()

In [ ]:
gc_star_ratio_table_incl_cpg.to_hdf('../results/gc_star_ratio_table_incl_cpg.h5', key='df', format='table')

In [ ]:
del gc_star
del gc_star_ratios
del gc_star_ratio_table
del gc_star_incl_cpg
del gc_star_ratios_incl_cpg
del gc_star_ratio_table_incl_cpg
gc.collect()

## GC* at all hotspots relative to flanks and hotspot strength (center_flank_data)

In [ ]:
gc_star = pd.read_hdf('../results/gc_star.h5')
gc_star.head()

In [ ]:
one_percent_gcstar_quantiles = pd.read_hdf('../results/one_percent_gcstar_quantiles.h5')
one_percent_gcstar_quantiles.head()

In [ ]:
hotspot_anno_dist = pd.read_hdf('../results/hotspot_anno_dist.h5')
hotspot_anno_dist.head()

In [ ]:
# hotspot_strengths = (pd.merge((gc_star_table.loc[gc_star_table.bin == 0]
#                                .groupby(['species', 'species_code'], observed=True)
#                                .GCstar.mean()
#                                .to_frame('GCstar_hot')
#                                .reset_index()
#                               ),
#                               (gc_star_table.loc[gc_star_table.bin.abs() >= 20000]
#                                .groupby(['species', 'species_code'], observed=True)
#                                .GCstar.mean()
#                                .to_frame('GCstar_flank')
#                                .reset_index()
#                               ),
#                               on=['species', 'species_code']
#                             )
#                     .merge(one_percent_gcstar_quantiles, on=['species', 'species_code'], how='left')
#                     )

# from scipy.special import logit
# hotspot_strengths['hotspot_strength'] = \
#     (logit(hotspot_strengths.GCstar_hot) - logit(hotspot_strengths.GCstar_mut)) / \
#     (logit(hotspot_strengths.GCstar_flank) - logit(hotspot_strengths.GCstar_mut))

# hotspot_strengths['log_hotspot_strength'] = np.log2(hotspot_strengths['hotspot_strength'])
# hotspot_strengths.head()

## GC* ratio hotspots overlapping and not overlapping CGIs (gc_star_rel_cgi)

In [ ]:
subst_data = pd.read_hdf('../results/subst_data.h5')
hotspot_anno_dist = pd.read_hdf('../results/hotspot_anno_dist.h5')

In [ ]:
df = (subst_data
      .merge(hotspot_anno_dist[['hotspot_center', 'dist_cgi']], 
             on='hotspot_center')
      .assign(cgi_overlapping=lambda df: df.dist_cgi==0)
     )

table = pd.pivot_table(df, index=['species', 'species_code', 'bin', 'cgi_overlapping'], 
                    columns='pattern', values='rate')

gc_star_rel_cgi_table = (table
                                .groupby(['species', 'species_code', 'bin', 'cgi_overlapping'], group_keys=False)
                                .apply(subst_gc_star)
                                .reset_index()
                               )

gc_star_rel_cgi = gc_star_rel_cgi_table.melt(id_vars=['species', 'species_code', 'cgi_overlapping', 'bin'], var_name='variable', value_name='frequency')
gc_star_rel_cgi.head()

In [ ]:
gc_star_rel_cgi.to_hdf('../results/gc_star_rel_cgi.h5', key='df', format='table')

In [ ]:
def compute_gc_star_ratios(df):
    flank = df.loc[(df.bin.abs() < 40000) & (df.bin.abs() >= 10000)].groupby(['species', 'cgi_overlapping', 'variable'], group_keys=False).frequency.mean()
    center = df.loc[df.bin == 0].groupby(['species', 'cgi_overlapping', 'variable']).frequency.mean()
    ratios = (center / flank).reset_index().rename(columns={'frequency': 'ratio'})
    ratios_df = pd.pivot_table(ratios, index=['species', 'cgi_overlapping'], columns='variable', values='ratio').rename(columns=str).reset_index()
    ratios = ratios_df.melt(id_vars=['species', 'cgi_overlapping'], var_name='variable', value_name='ratio')
    return ratios

gc_star_ratios_rel_cgi = compute_gc_star_ratios(gc_star_rel_cgi)
gc_star_ratios_rel_cgi.head()

In [ ]:
gc_star_ratios_rel_cgi.to_hdf('../results/gc_star_ratios_rel_cgi.h5', key='df', format='table')

In [ ]:
del gc_star_rel_cgi
del gc_star_ratios_rel_cgi
del hotspot_anno_dist
del subst_data
gc.collect()

## Center vs flank (indiv_subst_ratios)

In [ ]:
subst_data = pd.read_hdf('../results/subst_data.h5')

In [ ]:
subst_flank = subst_data.loc[(40000 > subst_data.bin.abs()) & (subst_data.bin.abs() >= 10000)].groupby(['species', 'pattern']).rate.mean()
subst_center = subst_data.loc[subst_data.bin == 0].groupby(['species', 'pattern']).rate.mean()
subst_ratios = (subst_center / subst_flank).reset_index().rename(columns={'rate': 'ratio'})
df = subst_ratios.pivot(index='species', columns='pattern', values='ratio').rename(columns=str).reset_index()
plot_df = df.melt(id_vars=['species'], var_name='pattern', value_name='ratio')
plot_df.head()

In [ ]:
order=['rA2C', 'rT2G', 'rA2G', 'rT2C', 'rC2G', 'rG2C', 'rA2T', 'rT2A', 'rC2T', 'rG2A', 'rC2A', 'rG2T']

Individual species with error bars:

In [ ]:
indiv_subst_flank = subst_data.loc[(40000 > subst_data.bin.abs()) & (subst_data.bin.abs() >= 10000)].groupby(['species', 'pattern', 'hotspot_center']).rate.mean()
indiv_subst_flank.name = 'rate_flank'
indiv_subst_center = subst_data.loc[subst_data.bin == 0].groupby(['species', 'pattern', 'hotspot_center']).rate.mean()
indiv_subst_center.name = 'rate_center'

In [ ]:
indiv_subst_ratios = (indiv_subst_center / indiv_subst_flank).reset_index().rename(columns={0: 'ratio'})
# ratios = (center / flank).reset_index().rename(columns={'rate': 'ratio'})#.dropna()
indiv_subst_ratios.head()

In [ ]:
indiv_subst_ratios.to_hdf('../results/indiv_subst_ratios.h5', key='df', format='table')

In [ ]:
del indiv_subst_flank
del indiv_subst_center
del indiv_subst_ratios
del subst_data
gc.collect()

# Integrated s-jump bias (integrated_bias)

In [ ]:
subst_data = pd.read_hdf('../results/subst_data.h5')

In [ ]:
df = subst_data.pivot(index=['species', 'species_code', 'hotspot_center', 'bin'], columns='pattern', values='rate')

df.columns = df.columns.astype('str')
for a, b in paired_patterns:
    df[f'{a}-{b}'] = df[a] - df[b]
df.head()

In [ ]:
plot_df = (df[['rT2G-rA2C', 'rA2G-rT2C', 'rA2T-rT2A',
               'rG2T-rC2A', 'rC2G-rG2C', 'rC2T-rG2A']]
           .reset_index()
           .melt(id_vars=['species', 'species_code', 'hotspot_center', 'bin'], value_name='rate')
          )
plot_df.head()

In [ ]:
def get_abs_area(df):
    left = np.trapz(0.5-df.loc[df.bin <= 0].rate, x=df.loc[df.bin <= 0].bin)
    right = np.trapz(df.loc[df.bin >= 0].rate-0.5, x=df.loc[df.bin >= 0].bin)
    return left + right

integrated_bias = (plot_df
                   .groupby(['species', 'species_code', 'hotspot_center', 'pattern'])
                   .apply(get_abs_area)
                   .to_frame('total_bias')
                   .reset_index()
                  )
integrated_bias.head()

In [ ]:
integrated_bias.to_hdf('../results/integrated_bias.hdf', key='df', format='table')

In [ ]:
integrated_bias.loc[integrated_bias.species_code == 'TAEGU'].groupby('pattern').total_bias.sum()

In [ ]:
del df
del plot_df
del integrated_bias
gc.collect()

In [ ]:
# def compute_gc_star(df):
    
#     GCflux = (df.rA2G + df.rA2C + df.rT2G + df.rT2C) / \
#              (df.rC2A + df.rC2T + df.rG2A + df.rG2T)
#     GCstar = GCflux/(1 + GCflux)
    
#     return pd.DataFrame(dict(hotspot_center=df.hotspot_center, GCstar=GCstar))

In [ ]:
# GCstar_center_by_hotspot = (df
#                        .reset_index()
#                        .loc[lambda df: df.bin.abs() <= 3000]
#                        .groupby(['species', 'species_code', 'hotspot_center'], group_keys=False)
#                        .apply(compute_gc_star)
#                        .reset_index(drop=True)
#                       )
# GCstar_center_by_hotspot.head()

In [ ]:
# GCstar_flank_by_hotspot = df.reset_index().loc[lambda df: df.bin.abs() >= 10000].groupby(['species', 'species_code', 'hotspot_center'], group_keys=False).apply(compute_gc_star).reset_index(drop=True)
# GCstar_flank_by_hotspot.head()

# Bias in base composition for comparison (comp_data)

In [ ]:
hotspot_data = pd.read_hdf('../results/hotspot_data.h5')
hotspot_anno_dist = pd.read_hdf('../results/hotspot_anno_dist.h5')

In [ ]:
df = (hotspot_data
      .loc[hotspot_data.bin.abs() <= 40000, 
           ['species', 'species_code', 'bin', 'nC', 'nT', 'nG', 'nA', 'hotspot_center']]
      .merge(hotspot_anno_dist[['hotspot_center', 'dist_cgi']], 
             on='hotspot_center')
      .loc[lambda df: (df.dist_cgi == 0) | (df.dist_cgi.abs() > 40000)]
      .assign(cgi_overlapping=lambda df: df.dist_cgi==0)
      .loc[:, ['species', 'species_code', 'bin', 'nC', 'nT', 'nG', 'nA', 'cgi_overlapping']]
     )
df['C-G'] = (df.nC-df.nG) / (df.nC + df.nG)
df['T-A'] = (df.nT-df.nA) / (df.nT + df.nA)
comp_data = df.melt(id_vars=['species', 'species_code', 'bin', 'cgi_overlapping'], value_vars=['C-G', 'T-A']).sort_values(by=['species', 'bin', 'variable'])
comp_data.head()

In [ ]:
comp_data.to_hdf('../results/comp_data.h5', key='df', format='table')

In [ ]:
del hotspot_data
del hotspot_anno_dist
del comp_data
gc.collect()